# Simplicity bias in topology-optimized mechanical metamaterials
SIMP optimization of a periodic cell for **shear**, **directional** and **auxetic** objectives; periodic homogenization; three complexity estimators (CLZ, zlib, BDM); matched random baselines; Cliff's δ; neutral sets; and the five figures.

Run order: **1 → 2 → 3** (engine), then **4** (config), then **5–10** (generate / analyse), then **11** (figures).
Generation checkpoints to `sweep_data/*.pkl` and is skipped if the files already exist.

## 1. Environment

In [1]:
# === RUN THIS FIRST
import os
for _v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS","VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")
%pip install -q numpy scipy joblib
%pip install -q pybdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


## 2. Physics engine
Periodic homogenization (energy method), SIMP interpolation, density filter.

In [2]:
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import splu
from scipy import ndimage

def _D(nu): return 1/(1-nu**2)*np.array([[1,nu,0],[nu,1,0],[0,0,(1-nu)/2]])
def _KE(nu):
    g=[0.5-0.5/np.sqrt(3),0.5+0.5/np.sqrt(3)];D=_D(nu);KE=np.zeros((8,8))
    for x in g:
        for y in g:
            dx=np.array([-(1-y),(1-y),y,-y]);dy=np.array([-(1-x),-x,x,(1-x)])
            B=np.zeros((3,8));B[0,0::2]=dx;B[1,1::2]=dy;B[2,0::2]=dy;B[2,1::2]=dx
            KE+=(B.T@D@B)*0.25
    return KE
_C=np.array([[0,0],[1,0],[1,1],[0,1]],float)
def _X0():
    X=np.zeros((8,3))
    for n,(xl,yl) in enumerate(_C):
        X[2*n,0]=xl;X[2*n+1,1]=yl;X[2*n,2]=0.5*yl;X[2*n+1,2]=0.5*xl
    return X

class Homogenizer:
    """2D periodic FE homogenizer (plane stress). Returns C* and, optionally,
    the per-element mutual-energy tensor Q for self-adjoint sensitivities."""
    def __init__(self,N,nu=0.3,penal=3,Emin=1e-9,E0=1.0):
        self.N=N;self.nu=nu;self.penal=penal;self.Emin=Emin;self.E0=E0
        self.KE=_KE(nu);self.X0=_X0();self.ndof=2*N*N
        nid=lambda r,c:(r%N)*N+(c%N)
        ed=np.zeros((N*N,8),int)
        for i in range(N):
            for j in range(N):
                nodes=[nid(i,j),nid(i,j+1),nid(i+1,j+1),nid(i+1,j)]
                ed[i*N+j]=np.array([[2*n,2*n+1] for n in nodes]).flatten()
        self.edofs=ed
        self.rows=np.repeat(ed,8,axis=1).reshape(-1);self.cols=np.tile(ed,(1,8)).reshape(-1)
        self.free=np.setdiff1d(np.arange(self.ndof),[0,1])
    def solve(self,xf,want_Q=False):
        N=self.N;Erho=self.Emin+(xf**self.penal)*(self.E0-self.Emin)
        ke=Erho.reshape(-1)[:,None,None]*self.KE[None]
        K=coo_matrix((ke.reshape(-1),(self.rows,self.cols)),shape=(self.ndof,)*2).tocsc()
        keX=np.einsum('eij,jk->eik',ke,self.X0);F=np.zeros((self.ndof,3))
        np.add.at(F,self.edofs.reshape(-1),keX.reshape(-1,3))
        chi=np.zeros((self.ndof,3));lu=splu(K[self.free][:,self.free]);chi[self.free]=lu.solve(F[self.free])
        dX=self.X0[None]-chi[self.edofs];Y=float(N*N)
        Cm=np.einsum('eai,eab,ebj->eij',dX,ke,dX)            # per-elem contributions
        C=Cm.sum(0)/Y
        if not want_Q: return C
        Q=np.einsum('eai,ab,ebj->eij',dX,self.KE,dX)         # unit-modulus mutual energy
        return C,Q

def periodic_filter_matrix(N,rmin):
    """Symmetric periodic density-filter kernel (cone weights, radius rmin)."""
    r=int(np.ceil(rmin)-1);xs=np.arange(-r,r+1);K=np.maximum(0,rmin-np.abs(np.add.outer(xs,xs*0))-0)  # placeholder
    yy,xx=np.meshgrid(xs,xs,indexing='ij');w=np.maximum(0,rmin-np.sqrt(xx**2+yy**2));w/=w.sum()
    return w
def pfilt(field,w): return ndimage.convolve(field,w,mode='wrap')

# ---------- objectives: return (minimise_value, per-component weights on dC) ----------
def obj_bulk (C): return -(C[0,0]+C[1,1]+2*C[0,1])/4, {'00':-0.25,'11':-0.25,'01':-0.5}
def obj_shear(C): return -C[2,2], {'22':-1.0}
def obj_c11  (C): return -C[0,0], {'00':-1.0}
def obj_aux  (C): return  C[0,1], {'01': 1.0}          # minimise C12 -> negative Poisson

def topopt(rho0,homog,objective,vol=0.4,rmin=1.5,maxit=120,tol=1e-3,move=0.2,verbose=False):
    """SIMP + OC inverse homogenisation from initial density rho0. Returns
    (converged continuous density, binary design @0.5, final C*)."""
    N=homog.N;w=periodic_filter_matrix(N,rmin)
    x=np.clip(rho0.astype(float),1e-3,1.0);x[:]=vol if x.mean()==0 else x*vol/x.mean()
    x=np.clip(x,1e-3,1.0)
    for it in range(maxit):
        xf=pfilt(x,w)
        C,Q=homog.solve(xf,want_Q=True)
        val,wts=objective(C)
        dE=homog.penal*xf.reshape(-1)**(homog.penal-1)*(homog.E0-homog.Emin)
        dc=np.zeros(N*N)
        for key,cw in wts.items():
            i,j=int(key[0]),int(key[1]);dc+=cw*dE*Q[:,i,j]/(N*N)
        dc=pfilt(dc.reshape(N,N),w).reshape(-1)               # chain rule through filter
        # OC update (minimisation: use -dc as the "compliance-like" gradient)
        dcn=np.minimum(0,dc)                                  # OC needs <=0; clip ascent dirs
        l1,l2=1e-9,1e9;xn=x.copy()
        while (l2-l1)/(l1+l2)>1e-4:
            lm=0.5*(l1+l2)
            be=np.sqrt(np.maximum(0,-dc)/lm)
            xn=np.clip(np.clip(x.reshape(-1)*be,x.reshape(-1)-move,x.reshape(-1)+move),1e-3,1.0).reshape(N,N)
            if pfilt(xn,w).mean()>vol: l1=lm
            else: l2=lm
        ch=np.abs(xn-x).max();x=xn
        if verbose and it%20==0: print(f"  it{it:3d} obj={val:+.5f} vol={pfilt(x,w).mean():.3f} ch={ch:.3f}")
        if ch<tol and it>10: break
    xf=pfilt(x,w);C=homog.solve(xf)
    return x,(xf>0.5).astype(np.uint8),C

## 3. Objectives and complexity
`obj_shear = -C33`, `obj_dirx = -C11`, `obj_aux = +C12` (the optimizer minimizes). 

In [3]:
# Objectives: each returns (value-to-MINIMISE, weights on C-components for sensitivities)
def obj_iso (C): return -(C[0,0]+C[1,1])/2, {'00':-0.5,'11':-0.5}   # isotropic stiffness
def obj_shear(C): return -C[2,2], {'22':-1.0}                        # shear modulus
def obj_dirx(C): return -C[0,0], {'00':-1.0}                         # directional (x) stiffness
def obj_aux (C): return  C[0,1], {'01': 1.0}                         # auxetic (minimise C12)
OBJECTIVES = {'iso':obj_iso, 'shear':obj_shear, 'dirx':obj_dirx, 'aux':obj_aux}

def score_all(C):
    """Performance of a design on every objective (higher = better)."""
    nu = C[0,1]/C[1,1] if abs(C[1,1])>1e-9 else 0.0
    return {'iso':(C[0,0]+C[1,1])/2, 'shear':C[2,2], 'dirx':C[0,0], 'aux':-nu}

In [4]:
"""
complexity.py -- Phase 2.
Canonical orientation (dihedral group D4: 4 rotations x 2 reflections) so that
symmetry-equivalent designs map to ONE output, plus complexity estimators.
"""
import zlib
import numpy as np


def canonical(b):
    """Map an NxN binary array to its canonical orientation: the lexicographically
    smallest packed-byte representation over the 8 dihedral transforms. Applied
    before complexity AND before basin counting, so rotations/reflections of the
    same design are treated as one output."""
    b = np.asarray(b, dtype=np.uint8)
    forms = []
    a = b
    for _ in range(4):
        forms.append(a)
        forms.append(np.fliplr(a))
        a = np.rot90(a)
    return min(forms, key=lambda f: np.packbits(np.ascontiguousarray(f).reshape(-1)).tobytes())


def key_of(b):
    """Hashable identity of a design's canonical orientation (for basin counts)."""
    return np.packbits(np.ascontiguousarray(canonical(b)).reshape(-1)).tobytes()


def k_zlib(b):
    """Raw-DEFLATE complexity proxy (bytes), header stripped via wbits=-15."""
    c = zlib.compressobj(9, zlib.DEFLATED, -15)
    packed = np.packbits(np.ascontiguousarray(b, dtype=np.uint8).reshape(-1)).tobytes()
    return float(len(c.compress(packed) + c.flush()))


_BDM = None
def k_bdm(b):
    """BDM complexity (pybdm) -- decomposes large arrays into CTM-valued blocks."""
    global _BDM
    if _BDM is None:
        from pybdm import BDM
        _BDM = BDM(ndim=2)
    return float(_BDM.bdm(np.asarray(b, dtype=int)))


def complexity(b, measure="zlib", canonicalize=True):
    """Complexity of a design under the chosen estimator, on its canonical orientation."""
    bc = canonical(b) if canonicalize else np.asarray(b, np.uint8)
    if measure == "zlib":
        return k_zlib(bc)
    if measure == "bdm":
        return k_bdm(bc)
    raise ValueError(f"unknown measure {measure!r}")

## 4. Engine functions

**Complexity estimators  (CLZ, zlib, BDM -> K)**

In [5]:

import math, itertools, numpy as np
from scipy.stats import mannwhitneyu, spearmanr
import warnings as _warnings
_warnings.filterwarnings("ignore", message="An input array is constant")

# ====================================================================== #
#  ESTIMATORS                                                            #
# ====================================================================== #
def _lz76(seq):
    """Lempel-Ziv (1976) production complexity c(x): number of phrases (Kaspar-Schuster)."""
    n = len(seq)
    if n == 0: return 0
    if n == 1: return 1
    i, c, l, k, kmax = 0, 1, 1, 1, 1
    while True:
        if seq[i + k - 1] == seq[l + k - 1]:
            k += 1
            if l + k > n:
                c += 1; break
        else:
            if k > kmax: kmax = k
            i += 1
            if i == l:
                c += 1; l += kmax
                if l + 1 > n: break
                i, k, kmax = 0, 1, 1
            else:
                k = 1
    return c

def clz(design):
    """2018 CLZ estimator (Eq.5) on the canonical orientation of a binary design."""
    s = np.ascontiguousarray(canonical(design)).reshape(-1).astype(np.int8)
    n = len(s)
    if np.all(s == s[0]):
        return math.log2(n)
    return math.log2(n) * 0.5 * (_lz76(s) + _lz76(s[::-1]))

def K(design, measure):
    if measure == "clz":
        return clz(design)
    return complexity(design, measure)   # 'zlib' or 'bdm'

**SIMP optimizer + parallel generation**

In [6]:
def _parallel(func, args, jobs):
    if not jobs or jobs == 1:
        return [func(a) for a in args]
    from joblib import Parallel, delayed
    return Parallel(n_jobs=jobs, backend="loky", batch_size="auto")(delayed(func)(a) for a in args)

def topopt_hardened(rho0, H, objective, vol, rmin, sched=(1.0,1.5,2.0,2.5,3.0),
                    iters_per=120, tol=1e-4, move=0.15, return_gray=False):
    N = H.N; w = periodic_filter_matrix(N, rmin)
    x = np.clip(rho0.astype(float), 1e-3, 1.0)
    x[:] = vol if x.mean() == 0 else x * vol / x.mean(); x = np.clip(x, 1e-3, 1.0)
    total = 0; ch = np.inf
    for p in sched:
        H.penal = p
        for it in range(iters_per):
            total += 1
            xf = pfilt(x, w); C, Q = H.solve(xf, want_Q=True); val, wts = objective(C)
            dE = H.penal * xf.reshape(-1) ** (H.penal - 1) * (H.E0 - H.Emin)
            dc = np.zeros(N * N)
            for key, cw in wts.items():
                i, j = int(key[0]), int(key[1]); dc += cw * dE * Q[:, i, j] / (N * N)
            dc = pfilt(dc.reshape(N, N), w).reshape(-1)
            l1, l2 = 1e-9, 1e9
            while (l2 - l1) / (l1 + l2) > 1e-4:
                lm = 0.5 * (l1 + l2); be = np.sqrt(np.maximum(0, -dc) / lm)
                xn = np.clip(np.clip(x.reshape(-1) * be, x.reshape(-1) - move, x.reshape(-1) + move),
                             1e-3, 1.0).reshape(N, N)
                if pfilt(xn, w).mean() > vol: l1 = lm
                else: l2 = lm
            ch = np.abs(xn - x).max(); x = xn
            if ch < tol and it > 10: break          # early-exit THIS stage when settled
    # FIX: convergence judged on the FINAL stage's last density change (ch), not on
    # whether the last stage early-exited. The old per-stage flag reported ~0 converged
    # for designs that were in fact settled.
    conv = bool(ch < tol)
    H.penal = sched[-1]; xf = pfilt(x, w); C = H.solve(xf)
    binar = (xf > 0.5).astype(np.uint8)
    if return_gray: return binar, C, conv, total, xf
    return binar, C, conv, total
def _worker(args):
    seed, objname, N, p, vol, rmin, nu, penal, sched, iters_per, tol, move = args
    rng = np.random.default_rng(seed)
    H = Homogenizer(N, nu=nu, penal=penal)
    r0 = (rng.random((N, N)) < p).astype(float)
    sch = sched if sched is not None else (penal,)
    xb, C, conv, iters = topopt_hardened(r0, H, OBJECTIVES[objname], vol, rmin,
                                         sched=sch, iters_per=iters_per, tol=tol, move=move)
    return dict(seed=seed, xb=xb, C=C, conv=conv, iters=iters)

def generate(objname, n, N=32, p=0.4, vol=0.4, rmin=1.5, nu=0.3, penal=3,
             continuation=True, iters_per=120, tol=1e-4, move=0.15, jobs=1, seed0=0):
    sched = (1.0, 1.5, 2.0, 2.5, 3.0) if continuation else None
    args = [(seed0 + s, objname, N, p, vol, rmin, nu, penal, sched, iters_per, tol, move)
            for s in range(n)]
    return _parallel(_worker, args, jobs)

_PH = {}

**Baselines: filtered-random, Bernoulli, coarse-blobs, correlation-matched**

In [7]:
def _valid_cell(rng, N, rmin, vol, w, H):
    from scipy.ndimage import gaussian_filter
    for _ in range(60):
        f = gaussian_filter(rng.random((N, N)), sigma=max(1.0, rmin), mode="wrap")
        thr = np.quantile(f, 1 - vol)
        xb = (pfilt((f >= thr).astype(float), w) > 0.5).astype(np.uint8)
        if 0.05 < xb.mean() < 0.95:
            return dict(xb=xb, C=H.solve(xb.astype(float)))
    return dict(xb=xb, C=H.solve(xb.astype(float)))

def filtered_random_cells(N, n, rmin, vol, seed=12345, nu=0.3):
    rng = np.random.default_rng(seed); w = periodic_filter_matrix(N, rmin)
    H = Homogenizer(N, nu=nu, penal=3)
    return [_valid_cell(rng, N, rmin, vol, w, H) for _ in range(n)]

def _bernoulli_cells(N, n, vol, rng, H):
    out = []
    while len(out) < n:
        xb = (rng.random((N, N)) < vol).astype(np.uint8)
        if 0.05 < xb.mean() < 0.95: out.append(dict(xb=xb, C=H.solve(xb.astype(float))))
    return out

def _coarse_cells(N, n, vol, rng, H, sigma):
    from scipy.ndimage import gaussian_filter
    out = []
    while len(out) < n:
        f = gaussian_filter(rng.random((N, N)), sigma=sigma, mode="wrap")
        thr = np.quantile(f, 1 - vol); xb = (f >= thr).astype(np.uint8)
        if 0.05 < xb.mean() < 0.95: out.append(dict(xb=xb, C=H.solve(xb.astype(float))))
    return out

def baseline_variants(N, n, rmin, vol, seed=999, nu=0.3):
    rng = np.random.default_rng(seed); H = Homogenizer(N, nu=nu, penal=3)
    return {"filtered(rmin)": filtered_random_cells(N, n, rmin, vol, seed=seed + 1, nu=nu),
            "raw-bernoulli":  _bernoulli_cells(N, n, vol, rng, H),
            "coarse-blobs":   _coarse_cells(N, n, vol, rng, H, sigma=max(3.0, 2 * rmin))}

def correlation_matched_cells(optima, N, n, vol, seed=2024, nu=0.3):
    """Phase-randomized surrogate matched to the optima's MEAN power spectrum (=> same
    two-point correlation function) and thresholded to the same density. If optima are still
    simpler than these, the simplicity lives in HIGHER-ORDER structure, not density or the
    two-point statistics -- the strongest 'simpler than random' control."""
    Od = _distinct(optima)
    P = np.zeros((N, N)); cnt = 0
    for d in Od:
        x = d["xb"].astype(float); x -= x.mean()
        P += np.abs(np.fft.fft2(x)) ** 2; cnt += 1
    if cnt == 0: return []
    amp = np.sqrt(P / cnt)
    rng = np.random.default_rng(seed); H = Homogenizer(N, nu=nu, penal=3)
    out = []; tries = 0
    while len(out) < n and tries < 80 * n:
        tries += 1
        field = np.real(np.fft.ifft2(amp * np.exp(1j * rng.uniform(0, 2 * np.pi, (N, N)))))
        xb = (field >= np.quantile(field, 1 - vol)).astype(np.uint8)
        if 0.05 < xb.mean() < 0.95: out.append(dict(xb=xb, C=H.solve(xb.astype(float))))
    return out

# --- (6) bootstrap CI on neutral-set degeneracy statistics ---
def compare_all_baselines(optima, baselines, objname, measures=("clz", "zlib", "bdm"), top_frac=0.2):
    Od = _distinct(optima); perf = np.array([score_all(d["C"])[objname] for d in Od], float)
    topq = np.quantile(perf, 1 - top_frac)
    Ktop = {m: np.array([K(d["xb"], m) for d in Od], float)[perf >= topq] for m in measures}
    by_b = {}
    for bname, bg in baselines.items():
        per = {}
        for m in measures:
            Kb = np.array([K(d["xb"], m) for d in bg], float)
            try: p = float(mannwhitneyu(Ktop[m], Kb, alternative="less")[1])
            except ValueError: p = float("nan")
            per[m] = dict(cliffs=float(_cliffs(Ktop[m], Kb)), p=p, mean_bg=float(Kb.mean()))
        by_b[bname] = per
    return dict(mean_top={m: float(Ktop[m].mean()) for m in measures}, by_baseline=by_b)

def report_all_baselines(res, alpha=0.05):
    print("\nBASELINE ROBUSTNESS  optima vs several random baselines (Cliff d<0 = optima simpler)")
    print("    Ktop: " + "  ".join(f"{m}={res['mean_top'][m]:.1f}" for m in res["mean_top"]))
    for bname, per in res["by_baseline"].items():
        seg = "  ".join(f"{m}:d={per[m]['cliffs']:+.2f}({'sig' if per[m]['p']<alpha and per[m]['cliffs']<0 else 'ns'})"
                        for m in per)
        print(f"    vs {bname:16s}: {seg}")
    print("    => robust if optima are significantly simpler vs ALL baselines; smallest |d| = most conservative test")


**Random candidate pool (for neutral sets)**

In [8]:
def _pool_worker(args):
    seed, N, rmin, vol, nu = args
    H = _PH.get((N, nu))
    if H is None: H = _PH[(N, nu)] = Homogenizer(N, nu=nu, penal=3)
    w = periodic_filter_matrix(N, rmin)
    return _valid_cell(np.random.default_rng(seed), N, rmin, vol, w, H)

def random_pool(N, M, rmin, vol, jobs=1, seed0=777, nu=0.3):
    """Candidate universe for H2: M random valid cells, each with effective tensor."""
    args = [(seed0 + i, N, rmin, vol, nu) for i in range(M)]
    return _parallel(_pool_worker, args, jobs)


**Statistics: canonical dedup + Cliff's delta**

In [9]:
def _distinct(designs):
    seen = {}; out = []
    for d in designs:
        k = key_of(d["xb"])
        if k not in seen: seen[k] = 1; out.append(d)
    return out

def _cliffs(a, b):
    # standard Cliff's delta = P(a>b) - P(a<b);  <0 => a tends SMALLER (simpler)
    d = np.asarray(a)[:, None] - np.asarray(b)[None, :]
    return (np.sum(d > 0) - np.sum(d < 0)) / (len(a) * len(b))

# ====================================================================== #
#  optima are simpler than matched random structures                     #
# ====================================================================== #
def complexity_vs_baseline(optima, background, objname, measures=("clz", "zlib", "bdm"),
              top_frac=0.2, n_boot=2000, alpha=0.05, rng=None):
    rng = rng or np.random.default_rng(0)
    Od = _distinct(optima)
    res = {"n_distinct_optima": len(Od), "n_background": len(background), "by_measure": {}}
    if len(Od) < 5 or len(background) < 5:
        res["error"] = "pool too small"; return res
    perf = np.array([score_all(d["C"])[objname] for d in Od], float)
    for m in measures:
        try:
            Ko = np.array([K(d["xb"], m) for d in Od], float)
            Kb = np.array([K(d["xb"], m) for d in background], float)
        except Exception as e:
            res["by_measure"][m] = {"error": f"{type(e).__name__}: {e}"}; continue
        topq = np.quantile(perf, 1 - top_frac); Ktop = Ko[perf >= topq]
        try: U, p_mw = mannwhitneyu(Ktop, Kb, alternative="less")
        except ValueError: p_mw = np.nan
        delta = _cliffs(Ktop, Kb)
        rho, p_sp = spearmanr(perf, Ko)
        bd = np.empty(n_boot); bs = np.empty(n_boot); nO, nB = len(Ko), len(Kb)
        for i in range(n_boot):
            iO = rng.integers(0, nO, nO); iB = rng.integers(0, nB, nB)
            pe, ko = perf[iO], Ko[iO]; tq = np.quantile(pe, 1 - top_frac)
            bd[i] = _cliffs(ko[pe >= tq] if (pe >= tq).sum() else ko, Kb[iB])
            r, _ = spearmanr(pe, ko); bs[i] = r if not np.isnan(r) else 0.0
        ci = lambda v: (float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2)))
        res["by_measure"][m] = dict(
            mean_top=float(Ktop.mean()), mean_rand=float(Kb.mean()),
            cliffs=float(delta), cliffs_CI=ci(bd), p_simpler=float(p_mw),
            spearman=float(rho), spearman_CI=ci(bs), p_spearman=float(p_sp))
    return res


def verdict(res, alpha=0.05):      # are the optima simpler than the baseline?
    if "error" in res: return "INCONCLUSIVE"
    ok = [(r["p_simpler"] < alpha and r["cliffs_CI"][1] < 0)
          for r in res["by_measure"].values() if "error" not in r]
    return "SUPPORTED" if (ok and all(ok)) else ("MIXED" if any(ok) else "NOT SUPPORTED")




**Neutral sets (degeneracy of the geometry->property map)**

In [10]:
def neutral_sets(cells, n_bins=12, comps=((0,0),(0,1),(2,2)), downsample=None):
    """How many DISTINCT topologies share an effective-property cell?
    cells: list of dict(xb,C). Bins the (C11,C12,C33) space into n_bins^3 and counts
    distinct canonical topologies per occupied cell. downsample=k pools the design into
    kxk blocks before keying, so near-duplicate shapes are not counted as distinct
    (a CONSERVATIVE under-count of degeneracy)."""
    if len(cells) < 50: return {"error": "pool too small (need >=50)"}
    C = np.array([d["C"] for d in cells])
    def dkey(xb):
        if downsample and downsample > 1:
            n = xb.shape[0] // downsample * downsample
            blk = xb[:n, :n].reshape(n//downsample, downsample, n//downsample, downsample)
            xb = (blk.mean(axis=(1,3)) > 0.5).astype(np.uint8)
        return key_of(xb)
    keys = [dkey(d["xb"]) for d in cells]
    feats = np.stack([C[:, i, j] for (i, j) in comps], axis=1)
    lo, hi = feats.min(0), feats.max(0); span = np.where(hi > lo, hi - lo, 1.0)
    idx = np.clip(((feats - lo) / span * n_bins).astype(int), 0, n_bins - 1)
    from collections import defaultdict
    topo = defaultdict(set); cnt = defaultdict(int)
    for r in range(len(cells)):
        b = tuple(idx[r]); topo[b].add(keys[r]); cnt[b] += 1
    sizes = np.array([len(topo[b]) for b in topo]); occ = np.array([cnt[b] for b in topo])
    multi = occ >= 3                         # only cells with enough samples to speak about
    s2 = sizes[multi] if multi.any() else sizes
    return dict(n_bins=n_bins, n_property_cells=int(len(topo)),
                mean_topologies=float(s2.mean()), median_topologies=float(np.median(s2)),
                max_topologies=int(s2.max()), frac_degenerate=float((s2 > 1).mean()),
                mean_samples_per_cell=float(occ.mean()))

def report_neutral_sets(ns_list):
    print("\nNEUTRAL SETS  degeneracy of the structure->property map")
    print("    distinct topologies sharing one effective-property cell (>1 => many shapes, same properties)")
    for label, ns in ns_list:
        if "error" in ns: print(f"    [{label}] {ns['error']}"); continue
        ci = ""
        if "frac_degenerate_CI" in ns:
            lo, hi = ns["frac_degenerate_CI"]; ci = f" [95% CI {100*lo:.0f}-{100*hi:.0f}%]"
        mci = ""
        if "median_topologies_CI" in ns:
            lo, hi = ns["median_topologies_CI"]; mci = f" [CI {lo:.0f}-{hi:.0f}]"
        print(f"    [{label:12s}] {ns['n_property_cells']:4d} property-cells | "
              f"degenerate (>1 topology): {100*ns['frac_degenerate']:3.0f}%{ci} | "
              f"topologies/cell median {ns['median_topologies']:.0f}{mci}, mean {ns['mean_topologies']:.1f}, max {ns['max_topologies']}")
    print("    => high 'topologies/cell' and 'degenerate %' across bin resolutions = strong neutral-set degeneracy")


**Physics validation**

In [11]:
def validate_physics(N, n_check=200, vol=0.4, nu=0.3, rng=None):
    rng = rng or np.random.default_rng(0); H = Homogenizer(N, nu=nu, penal=3)
    Csolid = H.solve(np.ones((N, N)))
    _d = 1.0 - nu * nu
    exact = np.array([[1/_d, nu/_d, 0.0], [nu/_d, 1/_d, 0.0], [0.0, 0.0, 1/(2*(1+nu))]])
    cells = filtered_random_cells(N, n_check, rmin=1.5, vol=vol, nu=nu)
    eps = rng.standard_normal((50, 3))
    sym, spd, voigt = [], [], []
    for d in cells:
        C = d["C"]; sym.append(float(np.max(np.abs(C - C.T))))
        spd.append(bool(np.all(np.linalg.eigvalsh(0.5 * (C + C.T)) > -1e-9)))
        v = float(d["xb"].mean())
        e_eff = np.einsum("ki,ij,kj->k", eps, C, eps); e_sol = np.einsum("ki,ij,kj->k", eps, Csolid, eps)
        voigt.append(bool(np.all(e_eff <= v * e_sol + 1e-6)))
    return dict(N=N, solid_max_err=float(np.max(np.abs(Csolid - exact))),
                max_asymmetry=float(np.max(sym)), spd_frac=float(np.mean(spd)),
                voigt_ok_frac=float(np.mean(voigt)), n_checked=len(cells))

def report_validation(v):
    print(f"\nPHYSICS VALIDATION (N={v['N']}, {v['n_checked']} random cells)")
    print(f"    exact solid-cell tensor error : {v['solid_max_err']:.2e}   (->0)")
    print(f"    tensor symmetry max|C-C^T|    : {v['max_asymmetry']:.2e}   (->0)")
    print(f"    positive-definite fraction    : {100*v['spd_frac']:.0f}%   (->100%)")
    print(f"    within Voigt upper bound      : {100*v['voigt_ok_frac']:.0f}%   (->100%)")
    ok = v['solid_max_err'] < 1e-3 and v['max_asymmetry'] < 1e-6 and v['spd_frac'] > 0.99 and v['voigt_ok_frac'] > 0.99
    print(f"    => physics {'PASSES' if ok else 'has a FLAG -- inspect above'}")


**Sweeps: mesh resolution and base Poisson ratio**

In [12]:
def resolution_sweep(n_list, n_ref=32, rmin_ref=1.5, scale_rmin=True, vol=0.4, n_optima=400,
                     measures=("clz", "zlib", "bdm"), continuation=True, iters_per=120,
                     tol=1e-4, jobs=1, save_dir="sweep_data", seed0=0):
    """For each grid N: validate physics, generate shear optima + a filter-matched baseline,
    (constant physical minimum feature size: rmin_N = rmin_ref * N / n_ref). Compare the
    scale-free Cliff's delta across N, NOT raw complexity K (which grows with grid size).
    Each N checkpoints separately and does not touch the main sweep files."""
    import os, pickle
    os.makedirs(save_dir, exist_ok=True)
    results = {}
    for N in n_list:
        rmin_N = rmin_ref * N / n_ref if scale_rmin else rmin_ref
        fp = os.path.join(save_dir, f"data_res_N{N}.pkl")
        if os.path.exists(fp):
            dd = pickle.load(open(fp, "rb"))
        else:
            val = validate_physics(N, n_check=100, vol=vol)
            opt = generate("shear", n_optima, N=N, vol=vol, rmin=rmin_N, continuation=continuation,
                           iters_per=iters_per, tol=tol, jobs=jobs, seed0=seed0)
            bg = filtered_random_cells(N, max(400, 2 * n_optima), rmin_N, vol)
            dd = dict(N=N, rmin=rmin_N, opt=opt, bg=bg, val=val)
            pickle.dump(dd, open(fp, "wb"))
        res = complexity_vs_baseline(dd["opt"], dd["bg"], "shear", measures=measures)
        results[N] = dict(delta=res, val=dd["val"], rmin=dd["rmin"], n_conv=sum(d["conv"] for d in dd["opt"]))
    return results

def report_resolution_sweep(results):
    print("\nRESOLUTION SWEEP -- is the effect mesh-independent? (compare Cliff's delta, not raw K)")
    for N, r in results.items():
        v = r["val"]; pok = v["solid_max_err"] < 1e-3 and v["spd_frac"] > 0.99 and v["voigt_ok_frac"] > 0.99
        segs = [f"{m}:d={rr['cliffs']:+.2f}(to {100*rr['mean_top']/max(rr['mean_rand'],1e-9):.0f}%)"
                for m, rr in r["delta"]["by_measure"].items() if "error" not in rr]
        print(f"\n  N={N:3d} (rmin={r['rmin']:.2f})  [physics {'ok' if pok else 'FLAG'}]  "
              f"distinct M={r['delta'].get('n_distinct_optima')}  converged={r['n_conv']}")
        print(f"     Cliff delta: {'  '.join(segs)}  ->  {verdict(r['delta'])}")
    a = [verdict(r["delta"]) for r in results.values()]
    print("\n  ----- across-resolution robustness -----")
    print(f"  => robust across N: {'YES (mesh-independent)' if all(x == 'SUPPORTED' for x in a) else 'resolution-dependent -- see above'}")
    print("  (raw K grows with N by construction; rmin scaled with N to hold the physical feature size fixed)")



# --- (1) multiplicity correction across the sweep family + primary endpoint ---
def poisson_sweep(nu_list, N=32, vol=0.4, rmin=1.5, n_optima=400, n_pool=4000,
                  measures=("clz", "zlib", "bdm"), continuation=True, iters_per=120,
                  tol=1e-4, jobs=1, save_dir="sweep_data", ns_bins=12, seed0=0):
    """For each base Poisson ratio nu: validate physics, generate shear optima + a
    Fresh generation per nu (cannot reuse nu=0.3 data); each nu checkpoints separately."""
    import os, pickle
    os.makedirs(save_dir, exist_ok=True)
    results = {}
    for nu in nu_list:
        fp = os.path.join(save_dir, f"data_nu_{nu}.pkl")
        if os.path.exists(fp):
            dd = pickle.load(open(fp, "rb"))
        else:
            val = validate_physics(N, n_check=100, vol=vol, nu=nu)
            opt = generate("shear", n_optima, N=N, vol=vol, rmin=rmin, nu=nu,
                           continuation=continuation, iters_per=iters_per, tol=tol, jobs=jobs, seed0=seed0)
            bg = filtered_random_cells(N, max(400, 2 * n_optima), rmin, vol, nu=nu)
            pool = random_pool(N, n_pool, rmin, vol, jobs=jobs, nu=nu)
            dd = dict(nu=nu, opt=opt, bg=bg, pool=pool, val=val)
            pickle.dump(dd, open(fp, "wb"))
        res = complexity_vs_baseline(dd["opt"], dd["bg"], "shear", measures=measures)
        ns = neutral_sets(dd["pool"], n_bins=ns_bins)
        results[nu] = dict(delta=res, ns=ns, val=dd["val"], n_conv=sum(d["conv"] for d in dd["opt"]))
    return results

def report_poisson_sweep(results):
    print("\nPOISSON-RATIO SWEEP (base material nu) -- Figure 4(c)")
    for nu, r in results.items():
        v = r["val"]; pok = v["solid_max_err"] < 1e-3 and v["spd_frac"] > 0.99 and v["voigt_ok_frac"] > 0.99
        print(f"\n  nu={nu:+.2f}  [physics {'ok' if pok else 'FLAG'}: solid_err={v['solid_max_err']:.1e}, "
              f"SPD={100*v['spd_frac']:.0f}%, Voigt={100*v['voigt_ok_frac']:.0f}%]")
        print(f"     verdict: {verdict(r['delta'])} | "
              f"distinct M={r['delta'].get('n_distinct_optima')} | converged={r['n_conv']}")
        ns = r["ns"]
        if "error" not in ns:
            print(f"     neutral sets: degenerate {100*ns['frac_degenerate']:.0f}%, "
                  f"topologies/cell median {ns['median_topologies']:.0f}, max {ns['max_topologies']}")
    a = [verdict(r["delta"]) for r in results.values()]
    print(f"\n  ----- across-nu robustness -----")
    print(f"  => robust across base Poisson ratio: "
          f"{'YES' if all(x == 'SUPPORTED' for x in a) else 'nu-dependent -- see above'}")



**Gray-fraction diagnostic**

In [13]:
def _worker_gray(args):
    seed, objname, N, p, vol, rmin, nu, penal, sched, iters_per, tol, move = args
    rng = np.random.default_rng(seed); H = Homogenizer(N, nu=nu, penal=penal)
    r0 = (rng.random((N, N)) < p).astype(float); sch = sched if sched is not None else (penal,)
    xb, C, conv, iters, xf = topopt_hardened(r0, H, OBJECTIVES[objname], vol, rmin,
                                             sched=sch, iters_per=iters_per, tol=tol, move=move, return_gray=True)
    return dict(seed=seed, xb=xb, xf=xf, C=C, conv=conv, iters=iters)

def generate_gray(objname, n, N=32, p=0.4, vol=0.4, rmin=1.5, nu=0.3, penal=3,
                  continuation=True, iters_per=120, tol=1e-4, move=0.15, jobs=1, seed0=0):
    sched = (1.0, 1.5, 2.0, 2.5, 3.0) if continuation else None
    args = [(seed0 + s, objname, N, p, vol, rmin, nu, penal, sched, iters_per, tol, move) for s in range(n)]
    return _parallel(_worker_gray, args, jobs)

**Estimator agreement**

In [14]:
def estimator_agreement(cells, measures=("clz", "zlib", "bdm"), max_cells=600):
    cs = cells[:max_cells]
    Kmat = {m: np.array([K(d["xb"], m) for d in cs], float) for m in measures}
    ms = list(measures); pairs = {}
    for i in range(len(ms)):
        for j in range(i + 1, len(ms)):
            r, p = spearmanr(Kmat[ms[i]], Kmat[ms[j]]); pairs[(ms[i], ms[j])] = dict(rho=float(r), p=float(p))
    return dict(n=len(cs), pairs=pairs, measures=ms)

def report_estimator_agreement(ea):
    print("\nESTIMATOR AGREEMENT  do the complexity measures rank designs consistently?")
    for (a, b), d in ea["pairs"].items():
        print(f"    {a:4s} vs {b:4s}: Spearman rho={d['rho']:+.2f} (p={d['p']:.1e})")
    rhos = [d["rho"] for d in ea["pairs"].values()]
    print(f"    => mean pairwise rho={np.mean(rhos):+.2f} across {ea['n']} designs "
          f"(high => estimators corroborate, they are NOT 3 independent oracles)")


## 5. Configuration

In [15]:
import os
JOBS         = os.cpu_count()
SWEEP_DIR    = "sweep_data"

# ---- physics / model ----
N            = 32           # cells per side
VOL          = 0.4          # solid fraction
CONTINUATION = True
ITERS_PER    = 120
TOL          = 1e-3         # density-field 'settled' threshold
MEASURES     = ("clz", "zlib", "bdm")

# ---- sweeps reported in the paper ----
RMIN_LIST    = (1.5, 2.0, 2.5, 3.0)      # feature size
N_LIST       = (16, 32, 64)              # mesh resolution
NU_LIST      = (-0.8, -0.5, -0.25, 0.0, 0.2, 0.3, 0.45)   # base Poisson
COND_RMIN    = RMIN_LIST[0]              # primary setting
N_REF        = 32                        # reference grid for rmin scaling

# ---- sample sizes ----
N_PRIMARY    = 2000    # shear runs at the primary rmin
N_OTHER      = 500     # runs per other objective
AP_POOL      = 15000   # random candidate pool (neutral sets)
BASELINE_N   = 800     # cells per baseline variant
N_RES_OPTIMA = 400     # optima per mesh resolution
NU_N_OPTIMA  = 400     # optima per Poisson ratio
NU_N_POOL    = 4000
NS_BINS      = (8, 12, 16)   # property-space bins for neutral sets

print(f"config | JOBS={JOBS} | N={N} vol={VOL} | rmin={RMIN_LIST} | measures={MEASURES}")

config | JOBS=12 | N=32 vol=0.4 | rmin=(1.5, 2.0, 2.5, 3.0) | measures=('clz', 'zlib', 'bdm')


## 6. Physics validation
Solid-cell tensor exact to ~1e-15; symmetry; positive-definiteness (Methods §2.1).

In [16]:
# ====================== PHYSICS VALIDATION ======================
# Sanity-check the homogeniser BEFORE trusting any verdict: exact solid-cell tensor,
# tensor symmetry, positive-definiteness, and the Voigt upper bound on random cells.
report_validation(validate_physics(N, n_check=200, vol=VOL))


PHYSICS VALIDATION (N=32, 200 random cells)
    exact solid-cell tensor error : 8.22e-15   (->0)
    tensor symmetry max|C-C^T|    : 1.94e-15   (->0)
    positive-definite fraction    : 100%   (->100%)
    within Voigt upper bound      : 100%   (->100%)
    => physics PASSES


## 7. Main generation
Optima for each objective + matched random background + candidate pool, per r_min.

In [17]:
import pickle, time
os.makedirs(SWEEP_DIR, exist_ok=True)
data_by_rmin = {}
t_all = time.time()
for rmin in RMIN_LIST:
    fp = os.path.join(SWEEP_DIR, f"data_rmin_{rmin}.pkl")
    if os.path.exists(fp):
        data_by_rmin[rmin] = pickle.load(open(fp, "rb")); print(f"rmin={rmin}: loaded checkpoint ({fp})"); continue
    t0 = time.time(); print(f"\n=== generating rmin={rmin} on {JOBS} cores ===")
    designs = {}
    for name in OBJECTIVES:
        nn = N_PRIMARY if name == "shear" else N_OTHER
        designs[name] = generate(name, nn, N=N, vol=VOL, rmin=rmin, continuation=CONTINUATION,
                                 iters_per=ITERS_PER, tol=TOL, jobs=JOBS)
        nc = sum(d["conv"] for d in designs[name])
        print(f"  [{name:5s}] {len(designs[name])} runs, {nc} converged ({100*nc/max(1,len(designs[name])):.0f}%)  [{time.time()-t0:.0f}s]")
    background     = filtered_random_cells(N, max(400, 4*N_OTHER), rmin=rmin, vol=VOL)
    candidate_pool = random_pool(N, AP_POOL, rmin=rmin, vol=VOL, jobs=JOBS)
    data_by_rmin[rmin] = dict(designs=designs, background=background, candidate_pool=candidate_pool)
    pickle.dump(data_by_rmin[rmin], open(fp, "wb"))
    print(f"  saved {fp}  |  rmin={rmin} done in {time.time()-t0:.0f}s")
print(f"\nALL rmin generated  |  total {time.time()-t_all:.0f}s")

rmin=1.5: loaded checkpoint (sweep_data\data_rmin_1.5.pkl)
rmin=2.0: loaded checkpoint (sweep_data\data_rmin_2.0.pkl)
rmin=2.5: loaded checkpoint (sweep_data\data_rmin_2.5.pkl)
rmin=3.0: loaded checkpoint (sweep_data\data_rmin_3.0.pkl)

ALL rmin generated  |  total 1s


## 8. Gray-fraction control
Are the optima essentially binary?

In [18]:
# ---- does the gray fraction drive the complexity result? (self-contained, fresh optima) ----
import numpy as np
_NDIAG  = 30                                  # lower for a faster check
_Ng, _volg, _rming = N, VOL, COND_RMIN
_rng = np.random.default_rng(0)
_opt_g = []
for _s in range(_NDIAG):
    _Hg = Homogenizer(_Ng)
    _r0 = (_rng.random((_Ng, _Ng)) < _volg).astype(float)
    _xb, _Cg, _convg, _itg, _xf = topopt_hardened(
        _r0, _Hg, OBJECTIVES["shear"], _volg, _rming,
        iters_per=ITERS_PER, tol=TOL, return_gray=True)
    _opt_g.append(dict(xb=_xb, C=_Cg, conv=_convg,
                       gray=float(np.mean((_xf > 0.1) & (_xf < 0.9)))))
_bg_g = filtered_random_cells(_Ng, 200, rmin=_rming, vol=_volg)

_gr = np.array([d["gray"] for d in _opt_g])
print(f"gray_frac over {_NDIAG} shear optima: mean={_gr.mean():.2f}  range=[{_gr.min():.2f}, {_gr.max():.2f}]  "
      f"converged={sum(d['conv'] for d in _opt_g)}/{_NDIAG}\n")

# (i) gray vs complexity
print("(i) does gray fraction correlate with measured complexity K?")
for m in MEASURES:
    _Ko = np.array([K(d["xb"], m) for d in _opt_g], float)
    _rho, _p = spearmanr(_gr, _Ko)
    tag = "negligible" if abs(_rho) < 0.3 else ("MODERATE" if abs(_rho) < 0.6 else "STRONG")
    print(f"    [{m:4s}] Spearman(gray, K) = {_rho:+.2f} (p={_p:.1e})  -> {tag}")

# (ii) the effect on ALL optima vs the CLEANEST (most binary) half
print("\n(ii) does the effect survive on the cleanest, most-binary optima?")
_med = float(np.median(_gr))
for _label, _sub in [("ALL optima", _opt_g),
                     (f"CLEANEST half (gray<={_med:.2f})", [d for d in _opt_g if d["gray"] <= _med])]:
    if len(_sub) < 5:
        print(f"    [{_label}] too few"); continue
    _h = complexity_vs_baseline(_sub, _bg_g, "shear", measures=MEASURES)
    _seg = "  ".join(f"{m}:d={_h['by_measure'][m]['cliffs']:+.2f}"
                     for m in MEASURES if "error" not in _h["by_measure"][m])
    print(f"    [{_label:28s}] Cliff d: {_seg}")
print("\nVERDICT: if (i) is negligible AND the cleanest-half Cliff d stays ~-1, the simplicity")
print("         result is NOT an artifact of thresholding gray.")

gray_frac over 30 shear optima: mean=0.21  range=[0.19, 0.30]  converged=0/30

(i) does gray fraction correlate with measured complexity K?
    [clz ] Spearman(gray, K) = +0.59 (p=6.1e-04)  -> MODERATE
    [zlib] Spearman(gray, K) = +0.61 (p=3.2e-04)  -> STRONG


C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\pybdm\utils.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


    [bdm ] Spearman(gray, K) = +0.27 (p=1.5e-01)  -> negligible

(ii) does the effect survive on the cleanest, most-binary optima?
    [ALL optima                  ] Cliff d: clz:d=-1.00  zlib:d=-1.00  bdm:d=-1.00
    [CLEANEST half (gray<=0.20)  ] Cliff d: clz:d=-1.00  zlib:d=-1.00  bdm:d=-1.00

VERDICT: if (i) is negligible AND the cleanest-half Cliff d stays ~-1, the simplicity
         result is NOT an artifact of thresholding gray.


## 9. Baselines
Filtered-random, raw-Bernoulli, coarse-blobs, correlation-matched.

In [19]:
# ============ BASELINE ROBUSTNESS  (Fig. 2c) — reuses the optima already generated ============
# "Simpler than random" is only as strong as "random". Compare the top shear optima
# against four baselines: filter-matched (FAIR -- same feature scale), raw-Bernoulli
# (maximally disordered), coarse-blobs (deliberately simple), and correlation-matched
# (same two-point correlation + density -- the strongest control). The defensible claim is
# "optima are simpler than random AT MATCHED FEATURE SCALE" (filter-matched row); coarse-blobs
# show how conservative the comparison can be pushed; corr-matched isolates higher-order structure.
_bl = baseline_variants(N, BASELINE_N, rmin=COND_RMIN, vol=VOL)
# (5) correlation-matched baseline: matches the optima's mean power spectrum (=> two-point
#     correlation) and density, with randomized phases. Simpler-than-THIS => the simplicity is
#     in higher-order structure, not density or two-point statistics.
_bl["corr-matched"] = correlation_matched_cells(data_by_rmin[COND_RMIN]["designs"]["shear"],
                                                N, BASELINE_N, VOL)
report_all_baselines(compare_all_baselines(data_by_rmin[COND_RMIN]["designs"]["shear"], _bl,
                                      "shear", measures=MEASURES))


BASELINE ROBUSTNESS  optima vs several random baselines (Cliff d<0 = optima simpler)
    Ktop: clz=381.8  zlib=82.3  bdm=333.0
    vs filtered(rmin)  : clz:d=-1.00(sig)  zlib:d=-1.00(sig)  bdm:d=-1.00(sig)
    vs raw-bernoulli   : clz:d=-1.00(sig)  zlib:d=-1.00(sig)  bdm:d=-1.00(sig)
    vs coarse-blobs    : clz:d=+0.05(ns)  zlib:d=+0.02(ns)  bdm:d=-1.00(sig)
    vs corr-matched    : clz:d=-1.00(sig)  zlib:d=-1.00(sig)  bdm:d=-1.00(sig)
    => robust if optima are significantly simpler vs ALL baselines; smallest |d| = most conservative test


## 10. Sweeps

In [20]:
res_sweep = resolution_sweep(N_LIST, n_ref=N_REF, rmin_ref=RMIN_LIST[0], scale_rmin=True,
                             vol=VOL, n_optima=N_RES_OPTIMA, measures=MEASURES,
                             continuation=CONTINUATION, iters_per=ITERS_PER, tol=TOL,
                             jobs=JOBS, save_dir=SWEEP_DIR)
report_resolution_sweep(res_sweep)


RESOLUTION SWEEP -- is the effect mesh-independent? (compare Cliff's delta, not raw K)

  N= 16 (rmin=0.75)  [physics ok]  distinct M=14  converged=398
     Cliff delta: clz:d=-0.72(to 89%)  zlib:d=-0.12(to 99%)  bdm:d=-1.00(to 42%)  ->  MIXED

  N= 32 (rmin=1.50)  [physics ok]  distinct M=398  converged=5
     Cliff delta: clz:d=-1.00(to 62%)  zlib:d=-1.00(to 72%)  bdm:d=-1.00(to 26%)  ->  SUPPORTED

  N= 64 (rmin=3.00)  [physics ok]  distinct M=1  converged=0
     Cliff delta:   ->  INCONCLUSIVE

  ----- across-resolution robustness -----
  => robust across N: resolution-dependent -- see above
  (raw K grows with N by construction; rmin scaled with N to hold the physical feature size fixed)


In [21]:
# ====================== POISSON-RATIO SWEEP (base material nu) ======================
pois = poisson_sweep(NU_LIST, N=N, vol=VOL, rmin=COND_RMIN, n_optima=NU_N_OPTIMA,
                     n_pool=NU_N_POOL, measures=MEASURES, continuation=CONTINUATION,
                     iters_per=ITERS_PER, tol=TOL, jobs=JOBS, save_dir=SWEEP_DIR)
report_poisson_sweep(pois)


POISSON-RATIO SWEEP (base material nu) -- Figure 4(c)

  nu=-0.80  [physics ok: solid_err=4.0e-14, SPD=100%, Voigt=100%]
     verdict: INCONCLUSIVE | distinct M=1 | converged=0
     neutral sets: degenerate 100%, topologies/cell median 8, max 3565

  nu=-0.50  [physics ok: solid_err=7.5e-15, SPD=100%, Voigt=100%]
     verdict: INCONCLUSIVE | distinct M=1 | converged=0
     neutral sets: degenerate 100%, topologies/cell median 9, max 3618

  nu=-0.25  [physics ok: solid_err=2.0e-14, SPD=100%, Voigt=100%]
     verdict: INCONCLUSIVE | distinct M=1 | converged=0
     neutral sets: degenerate 100%, topologies/cell median 10, max 3630

  nu=+0.00  [physics ok: solid_err=0.0e+00, SPD=100%, Voigt=100%]
     verdict: INCONCLUSIVE | distinct M=1 | converged=0
     neutral sets: degenerate 100%, topologies/cell median 8, max 3594

  nu=+0.20  [physics ok: solid_err=1.2e-14, SPD=100%, Voigt=100%]
     verdict: SUPPORTED | distinct M=97 | converged=0
     neutral sets: degenerate 100%, topologies/

**Estimator agreement**

In [22]:
# Do the three estimators rank designs consistently?
report_estimator_agreement(estimator_agreement(data_by_rmin[COND_RMIN]["candidate_pool"],
                                               measures=MEASURES, max_cells=800))


ESTIMATOR AGREEMENT  do the complexity measures rank designs consistently?
    clz  vs zlib: Spearman rho=+0.71 (p=1.0e-124)
    clz  vs bdm : Spearman rho=+0.60 (p=1.4e-78)
    zlib vs bdm : Spearman rho=+0.54 (p=9.5e-62)
    => mean pairwise rho=+0.62 across 800 designs (high => estimators corroborate, they are NOT 3 independent oracles)
